# 09 — Nonlinear programs with `nlpsol`

**Objectives.** Encode decision variables, parameters, objective, constraints, bounds, and initial guesses at the low level. The canonical form is $\min_x f(x,p)$ subject to $l_g\le g(x,p)\le u_g$ and $l_x\le x\le u_x$. Bounds are data, not symbolic inequalities.


In [1]:
# Make repository-local helpers importable without installing a package.
import sys
from pathlib import Path

repository_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

# Autoreload picks up edits to tutorial models before each notebook cell runs.
ipython = get_ipython()
if ipython is not None:
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

### Formulate and solve a parameterized nonlinear program

This cell uses the low-level `nlpsol` interface. The NLP dictionary maps
`x` to the decision vector, `p` to fixed solve-time data, `f` to the scalar
objective, and `g` to the constraint vector. IPOPT receives separate lower
and upper bounds for both decisions and constraints. Infinite entries
represent an open side of a bound.

For target parameter $p$, the problem is

$$
\begin{aligned}
\min_{x_0,x_1}\quad
    &(x_0-p)^2+(x_1-1)^2\\
\text{s.t.}\quad
    &x_0+x_1\ge 1,\\
    &x_0^2+x_1^2\le 4,\\
    &0\le x_0,x_1\le 3.
\end{aligned}
$$

The returned dictionary contains the primal optimum, objective value, and
dual multipliers such as `lam_g`. `solver.stats()` reports convergence and
the plugin-specific return status.


In [2]:
import casadi as ca
import numpy as np

x = ca.MX.sym("x", 2)
target = ca.MX.sym("target")
objective = (x[0] - target) ** 2 + (x[1] - 1) ** 2
constraints = ca.vertcat(
    x[0] + x[1],
    x[0] ** 2 + x[1] ** 2,
)

# nlpsol uses x for the NLP decision vector and p for fixed solve-time data.
nonlinear_program = {
    "x": x,
    "p": target,
    "f": objective,
    "g": constraints,
}
solver = ca.nlpsol(
    "solver",
    "ipopt",
    nonlinear_program,
    {"ipopt.print_level": 0, "print_time": False},
)

solution = solver(
    x0=[0.5, 0.5],
    p=2.0,
    lbx=[0, 0],
    ubx=[3, 3],
    lbg=[1, -ca.inf],
    ubg=[ca.inf, 4],
)
x_optimal = np.asarray(solution["x"].full()).ravel()
print(
    "optimal x:",
    x_optimal,
    "objective:",
    float(solution["f"]),
    "constraint multipliers:",
    solution["lam_g"],
)
print("solver status:", solver.stats()["return_status"])
assert solver.stats()["success"]
assert np.isfinite(x_optimal).all()


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

optimal x: [1.78885439 0.89442719] objective: 0.05572808782376574 constraint multipliers: [-1.4855e-09, 0.118034]
solver status: Solve_Succeeded


Constraint row order must match `lbg`, `ubg`, and `lam_g`. Multipliers help interpret active constraints and construct KKT residuals. Always check `solver.stats()['success']` and `return_status`; receiving an `x` array alone does not establish success.

**Common mistakes:** using an infeasible initial point without realizing it, unscaled variables, NaN-generating expressions, and treating a local result as globally optimal. **Recap:** formulate once, change numeric parameters and bounds between calls.
